# Session 4 - Indexes and Query Optimization Basics

Welcome. In this session we will learn:

- what an **index** is for (and when it helps)
- how to read **`EXPLAIN`**
- how **`EXPLAIN ANALYZE`** shows real timings
- **sargable** filters (filters that can use an index cleanly)

Sample tables are in database **`training`** on **your** MySQL. Use names like `training.customers`.


---
# Chapter 1 - What is an index?

## Phone-book story

Finding "Sharma" in a phone book:

```
No index (table scan)              With an index (lookup)
----------------------             ----------------------
Open page 1, scan names            Open index: S -> page 412
Open page 2, scan names            Go straight to page 412
... keep going ...                 Done
```

MySQL stores table rows on disk. An **index** is a sorted shortcut (usually a B-tree) that points to matching rows.

## Diagram: table vs index

```
training.orders (table heap / data)     idx on customer_id (sorted shortcut)
+-----------+--------+--------+         +-------------+-----------+
| order_id  | cust   | date   |         | customer_id | -> row    |
+-----------+--------+--------+         +-------------+-----------+
| 10248     | 1001   | 2023.. |         | 1001        | -> 10248  |
| 10249     | 2044   | 2023.. |         | 1001        | -> 11002  |
| 10250     | 1001   | 2023.. |         | 1002        | -> 10255  |
+-----------+--------+--------+         +-------------+-----------+

Query: WHERE customer_id = 1001
  -> use index, fetch only matching order rows
```

## Trade-off (keep this small)

| Upside | Cost |
|--------|------|
| Faster filters / joins on that column | Insert/update/delete must update the index |
| Less data read when selective | Uses disk space |

**When indexes help**

| Filter style | Typical plan | Why |
|--------------|--------------|-----|
| Selective (`customer_id = 1001`) | Index lookup | few matching rows |
| Broad (`freight > 50`) | Often full scan | too many rows still match |

Selective filters love indexes. Broad filters may still scan the table - and that can be the right choice.


---
# Chapter 2 - Indexes already on these tables

Useful indexes were created when the lab database was built (keys like `customer_id`, `order_date`, and foreign keys).

We **feel** them through `EXPLAIN` / `EXPLAIN ANALYZE` next - no need to invent new permanent indexes on `training.*`.


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - Indexes</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-indexes-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-indexes-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 3 - Reading EXPLAIN

`EXPLAIN` shows the **plan** MySQL chooses - it does **not** run the full query just to time it.

Think of it as asking the GPS for the route **before** you drive.

A typical row looks like this (shortened):

```
id | select_type | table | type | possible_keys | key | rows | Extra
1  | SIMPLE      | orders| ref  | idx_order_date| idx_order_date | 57 | NULL
```

You do not need every column on day one. Start with **`type`**, **`key`**, **`rows`**, then **`Extra`**.

## Column guide (what each field means)

| Column | Everyday meaning | What good / careful looks like |
|--------|------------------|--------------------------------|
| `id` | Step number in the plan (1, 2, ...) | Usually `1` for a simple single-table query |
| `select_type` | Kind of SELECT (`SIMPLE`, `PRIMARY`, subquery, ...) | `SIMPLE` is normal for class queries |
| `table` | Which table (or derived table) this row is about | Should match the table you expect |
| `partitions` | Partition pruning (advanced) | Often `NULL` in this course - safe to ignore for now |
| `type` | **How** MySQL reads rows | Prefer `const` / `ref` / `range` over `ALL` when filters are selective |
| `possible_keys` | Indexes MySQL *could* consider | A hint list - not the final choice |
| `key` | Index MySQL **actually chose** | Index name = used an index; `NULL` = no index for this access |
| `key_len` | How many bytes of that index are used | Larger can mean more columns of a composite index are used |
| `ref` | What is compared to the index (`const`, column name, ...) | `const` often means a literal like `customer_id = 1001` |
| `rows` | **Estimate** of rows MySQL thinks it will examine | Smaller is usually better for selective lookups (estimate, not exact) |
| `filtered` | Rough % of those rows kept after the condition | 100 means "most examined rows may qualify" (also an estimate) |
| `Extra` | Extra notes about the plan | See common values below |

## `type` values you will see most

| `type` | Plain English | Classroom example |
|--------|---------------|-------------------|
| `const` | At most one matching row (often primary key / unique) | `WHERE customer_id = 1001` |
| `ref` | Non-unique index lookup (many rows possible) | `WHERE order_date = '2023-06-15'` with a date index |
| `range` | Index used for a range of values | `WHERE order_date >= ... AND order_date < ...` |
| `index` | Walks an index (not always bad) | Sometimes for covering-index style access |
| `ALL` | Full table scan - read the table | Broad filters like `freight > 50` often land here |

```
Tighter access (usually)          Broader access
const -> ref -> range             index -> ALL
```

None of these is always wrong. A full scan (`ALL`) can be the right plan for small tables or filters that match most rows.

## `possible_keys` vs `key`

```
possible_keys  =  "candidates MySQL considered"
key            =  "the one it picked"  (or NULL)
```

If `possible_keys` lists an index but `key` is `NULL`, MySQL decided a scan was cheaper (or the filter was not selective enough).

## Common `Extra` notes

| Extra text | Meaning |
|------------|---------|
| `Using where` | After finding rows, MySQL still applies a WHERE condition |
| `Using index` | Answer came from the index alone (covering) - did not need full table rows |
| `Using filesort` | Extra sort step (not always bad; watch it on large results) |
| `Using temporary` | Built a temp table for this step (worth noticing on large data) |
| `NULL` / empty | Nothing special called out |

## Quick read order (use this on every plan)

1. **`table`** - which table?
2. **`key`** - index used, or `NULL`?
3. **`type`** - tight (`const`/`ref`/`range`) or `ALL`?
4. **`rows`** - small estimate or huge?
5. **`Extra`** - any `Using where` / `Using index` notes?

Then compare the three examples below: primary-key lookup, indexed date, broad freight filter.


### Lookup by primary key (expect a tight plan)


**Story:** Look up one customer by id - like opening one locker with the exact key.


In [1]:
EXPLAIN
SELECT *
FROM training.customers
WHERE customer_id = 1001;


+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+
| id | select_type | table     | partitions | type  | possible_keys | key     | key_len | ref   | rows | filtered | Extra |
+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+
|  1 | SIMPLE      | customers | NULL       | const | PRIMARY       | PRIMARY | 4       | const |    1 |   100.00 | NULL  |
+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+


### Filter on an indexed date column


**Use case:** "Show me orders on one date." An index on `order_date` can jump to that date.


In [2]:
EXPLAIN
SELECT order_id, customer_id, order_date, status
FROM training.orders
WHERE order_date = DATE '2023-06-15';


+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+
| id | select_type | table  | partitions | type | possible_keys         | key                   | key_len | ref   | rows | filtered | Extra |
+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+
|  1 | SIMPLE      | orders | NULL       | ref  | idx_orders_order_date | idx_orders_order_date | 3       | const |   57 |   100.00 | NULL  |
+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+


### Broad filter (often a table scan)


**Story:** "Show freight over 50" may match a huge share of orders.
Scanning the table can be cheaper than bouncing through an index for almost every row.


In [3]:
EXPLAIN
SELECT order_id, freight
FROM training.orders
WHERE freight > 50;


+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+
| id | select_type | table  | partitions | type | possible_keys | key  | key_len | ref  | rows  | filtered | Extra       |
+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+
|  1 | SIMPLE      | orders | NULL       | ALL  | NULL          | NULL | NULL    | NULL | 38996 |    33.33 | Using where |
+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+


### What to look for in the plan

Use the column guide above, then check:

1. Does **`key`** show an index name, or `NULL`?
2. Is **`type`** a tight access (`const` / `ref` / `range`) or `ALL`?
3. Is **`rows`** small or huge?
4. Does **`Extra`** say `Using index` (nice) or only a wide scan?

Compare the three plans above:

| Example | What you usually hope to see |
|---------|------------------------------|
| Primary key lookup | `type=const` (or similar), `key=PRIMARY`, `rows` near 1 |
| Indexed date filter | `type=ref` or `range`, `key` = the date index |
| Broad freight filter | often `type=ALL`, `key=NULL`, large `rows` |

That contrast is the main lesson of this chapter.


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - EXPLAIN</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-explain-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-explain-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 4 - EXPLAIN ANALYZE (timed run)

## The story (map vs drive)

| Tool | What it does | Everyday picture |
|------|--------------|------------------|
| `EXPLAIN` | Shows the **planned** route | Ask GPS for directions (estimates) |
| `EXPLAIN ANALYZE` | **Runs** the query, then reports real cost | Drive the route, then see how long each turn took |

```
EXPLAIN            ->  estimated plan (does not time the full work the same way)
EXPLAIN ANALYZE    ->  execute + measure (actual timings / row counts in the tree)
```

Use **`EXPLAIN`** while learning the *shape* of a plan (`type`, `key`, `rows`).  
Use **`EXPLAIN ANALYZE`** when you want **evidence** on your laptop data: "Was the tight plan actually fast?"

> Needs MySQL **8.0.18+**. Most current installs support it.

## Important differences from plain EXPLAIN

| Topic | `EXPLAIN` | `EXPLAIN ANALYZE` |
|-------|-----------|-------------------|
| Runs the query? | Plan only (no full timed execution story) | **Yes** - it executes the statement to measure |
| Output style | Table of columns (`type`, `key`, `rows`, ...) | **Tree** of operators with timing text |
| `rows` meaning | Estimate | You also see **actual** rows examined / produced |
| Best for | "Which index might it use?" | "How expensive was it really?" |
| Caution | Cheap to run often | Can be slower on heavy queries - it does the work |

Class tip: start with `EXPLAIN`. When two plans look different, confirm with `EXPLAIN ANALYZE`.

## How to read the tree (what to hunt for)

MySQL prints a nested plan. You do not need every word. Scan for:

| Phrase / idea | Meaning |
|---------------|---------|
| `->` tree lines | Steps of the plan (children feed parents) |
| `cost=...` | Optimizer cost estimate for that step |
| `actual time=...` | Real time measured while running (often `first_row..last_row`) |
| `rows=...` (estimate) vs actual rows | Compare "thought" vs "did" |
| Index / table lookup names | Confirms whether an index path was used |
| Filter / Table scan wording | Full scan vs index access |

A tiny mental checklist:

1. Did it use an **index lookup** or a **table scan**?
2. Are **actual times** tiny (microseconds/ms) or large?
3. Do **actual rows** stay small for a selective filter?

## When to prefer which

| Situation | Prefer |
|-----------|--------|
| Learning `type` / `key` / `rows` | `EXPLAIN` |
| Comparing "indexed date" vs "broad freight" cost | both - shape then timing |
| Checking a rewrite (sargable range vs `EXTRACT(YEAR...)`) | `EXPLAIN ANALYZE` on both |
| Huge report query in production | be careful - ANALYZE executes it |

## What you will compare next

Run the same three questions as Chapter 3, but with timings:

1. Primary-key lookup (should be very cheap)
2. Indexed date filter (usually cheap)
3. Broad freight filter (often more work - scan many rows)

Keep the numbers in mind: **fast + few rows** vs **slower + many rows**.


### EXPLAIN ANALYZE for a primary-key lookup


In [4]:
EXPLAIN ANALYZE
SELECT *
FROM training.customers
WHERE customer_id = 1001;


+---------------------------------------------------------------------------------------------------+
| EXPLAIN                                                                                           |
+---------------------------------------------------------------------------------------------------+
| -> Rows fetched before execution  (cost=0..0 rows=1) (actual time=355e-6..390e-6 rows=1 loops=1)
 |
+---------------------------------------------------------------------------------------------------+


### EXPLAIN ANALYZE for an indexed date filter


In [5]:
EXPLAIN ANALYZE
SELECT order_id, customer_id, order_date, status
FROM training.orders
WHERE order_date = DATE '2023-06-15';


+--------------------------------------------------------------------------------------------------------------------------------------------------------+
| EXPLAIN                                                                                                                                                |
+--------------------------------------------------------------------------------------------------------------------------------------------------------+
| -> Index lookup on orders using idx_orders_order_date (order_date=DATE'2023-06-15')  (cost=19.9 rows=57) (actual time=0.0499..0.0914 rows=57 loops=1)
 |
+--------------------------------------------------------------------------------------------------------------------------------------------------------+


### EXPLAIN ANALYZE for a broad freight filter


**Story:** Same broad filter as before (`freight > 50`).

Watch for a **table scan** style step and a larger **actual time** / row count than the PK and date lookups.


In [6]:
EXPLAIN ANALYZE
SELECT order_id, freight
FROM training.orders
WHERE freight > 50;


+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| EXPLAIN                                                                                                                                                                                                     |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| -> Filter: (orders.freight > 50.00)  (cost=3940 rows=12997) (actual time=0.066..8.94 rows=40000 loops=1)
    -> Table scan on orders  (cost=3940 rows=38996) (actual time=0.0633..4.86 rows=40000 loops=1)
 |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

### How to compare the three ANALYZE results

After you run the cells above, ask:

1. **Primary key** - is actual time tiny and rows about 1?
2. **Indexed date** - still relatively small time / row count?
3. **Broad freight** - larger actual time and many more rows examined?

That is the whole point of `EXPLAIN ANALYZE`: the plan story from Chapter 3 becomes a **timed** story on your data.

```
Selective lookup     ->  small actual time, small actual rows
Broad scan           ->  larger actual time, many rows touched
```

If a selective filter still looks expensive, re-check `WHERE` shape (Chapter 5 - sargable filters).


---
# Chapter 5 - Sargable filters

**Sargable** means the filter is written so MySQL can use an index on the column.

## Door-lock story

```
Good (sargable):                 Awkward (often weaker for indexes):
Leave the key as-is              Wrap the key in tape first
order_date >= '2023-01-01'       EXTRACT(YEAR FROM order_date) = 2023
AND order_date < '2024-01-01'
```

Wrapping `order_date` in a function makes it harder to use a plain date index as a tight range.

## Practical patterns

1. Filter on columns that already have indexes when you can.
2. Prefer ranges on the bare column over `EXTRACT` / `YEAR(column)` in `WHERE`.
3. Select only the columns you need.
4. Use `LIMIT` while exploring.
5. Join on keys (`customer_id`, `order_id`, ...) - indexes help joins too.


### Sargable date range (can use the date index as a range)


In [7]:
EXPLAIN
SELECT COUNT(*)
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2024-01-01';


+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+
| id | select_type | table  | partitions | type  | possible_keys         | key                   | key_len | ref  | rows  | filtered | Extra                    |
+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+
|  1 | SIMPLE      | orders | NULL       | range | idx_orders_order_date | idx_orders_order_date | 3       | NULL | 19498 |   100.00 | Using where; Using index |
+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+


### Year extract in WHERE (weaker for the same question)


Same business question ("orders in 2023"), different filter shape.
Compare `type` / `key` / `rows` with the range version above.


In [8]:
EXPLAIN
SELECT COUNT(*)
FROM training.orders
WHERE EXTRACT(YEAR FROM order_date) = 2023;


+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+
| id | select_type | table  | partitions | type  | possible_keys | key                   | key_len | ref  | rows  | filtered | Extra                    |
+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+
|  1 | SIMPLE      | orders | NULL       | index | NULL          | idx_orders_order_date | 3       | NULL | 38996 |   100.00 | Using where; Using index |
+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+


### Same comparison with EXPLAIN ANALYZE


Run both shapes with timings. Prefer the range form when you need a year filter and have an `order_date` index.


In [9]:
EXPLAIN ANALYZE
SELECT COUNT(*)
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2024-01-01';


+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| EXPLAIN                                                                                                                                                                                                                                                                                                                                                                                                                                                     |
+-------------------------------------------------------------------------------------------------------

In [10]:
EXPLAIN ANALYZE
SELECT COUNT(*)
FROM training.orders
WHERE EXTRACT(YEAR FROM order_date) = 2023;


+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| EXPLAIN                                                                                                                                                                                                                                                                                                                                                     |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

---
# Chapter 6 - Indexes with a TEMPORARY table

Create a small **temporary** copy, add an index, and run `EXPLAIN`.

Temporary tables disappear when the session ends - safe on your laptop.

> If you re-run this cell in the **same** session and see "already exists", reconnect (or open a new DBeaver SQL editor tab) and run it once.


### Temporary table + index + EXPLAIN / EXPLAIN ANALYZE


Build a small temp copy, add an index, then inspect the plan (and a timed plan) in the **same** session.


In [11]:
CREATE TEMPORARY TABLE temp_orders AS
SELECT order_id, customer_id, order_date, status, freight
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2023-04-01';

CREATE INDEX temp_orders_customer_id_idx
  ON temp_orders (customer_id);

EXPLAIN
SELECT *
FROM temp_orders
WHERE customer_id = 1001;

EXPLAIN ANALYZE
SELECT *
FROM temp_orders
WHERE customer_id = 1001;


+----+-------------+-------------+------------+------+-----------------------------+-----------------------------+---------+-------+------+----------+-------+
| id | select_type | table       | partitions | type | possible_keys               | key                         | key_len | ref   | rows | filtered | Extra |
+----+-------------+-------------+------------+------+-----------------------------+-----------------------------+---------+-------+------+----------+-------+
|  1 | SIMPLE      | temp_orders | NULL       | ref  | temp_orders_customer_id_idx | temp_orders_customer_id_idx | 5       | const |    1 |   100.00 | NULL  |
+----+-------------+-------------+------------+------+-----------------------------+-----------------------------+---------+-------+------+----------+-------+
+--------------------------------------------------------------------------------------------------------------------------------------------------------+
| EXPLAIN                                         

<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - Sargable filters</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-sargable-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-sargable-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>
